In [1]:
import numpy as np
import mrcfile
import time
import ot
from scipy.stats import binned_statistic


In [2]:
# Load 2 volumes
with mrcfile.open('128_org/000.mrc', permissive=True) as mrc:
    vol1 = mrc.data

with mrcfile.open('128_org/001.mrc', permissive=True) as mrc:
    vol2 = mrc.data

# Create positions once
positions = np.array([[i, j, k] 
                     for i in range(128) 
                     for j in range(128) 
                     for k in range(128)])

print(f"Volumes loaded: {vol1.shape}")
print(f"Positions: {positions.shape}")

p = vol1.flatten() / vol1.sum()
q = vol2.flatten() / vol2.sum()


Volumes loaded: (128, 128, 128)
Positions: (2097152, 3)


In [3]:

def sliced_sinkhorn_3d(vol1, vol2, n_projections=50, reg=0.01, n_bins=512):
    """
    Computes Sliced-Sinkhorn divergence between two 3D grid distributions.
    
    Parameters:
    - vol1, vol2: 3D numpy arrays (e.g., shape 128, 128, 128)
    - n_projections: Number of random directions to project onto.
    - reg: Entropic regularization parameter (lambda).
    - n_bins: Number of bins for the 1D histograms (controls resolution/speed).
    """
    # 1. Coordinate setup
    shape = vol1.shape
    x, y, z = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), np.arange(shape[2]), indexing='ij')
    coords = np.stack([x.ravel(), y.ravel(), z.ravel()], axis=-1).astype(np.float32)
    
    # Flatten volumes and normalize to unit mass (distributions)
    w1 = vol1.ravel().astype(np.float32)
    w2 = vol2.ravel().astype(np.float32)
    w1 /= (w1.sum() + 1e-12)
    w2 /= (w2.sum() + 1e-12)

    # 2. Generate random projection directions (unit vectors)
    projections = np.random.randn(n_projections, 3)
    projections /= np.linalg.norm(projections, axis=1, keepdims=True)

    total_dist = 0.0

    for i in range(n_projections):
        theta = projections[i]
        
        # 3. Project 3D coordinates to 1D
        proj_coords = coords @ theta
        
        # 4. Create 1D histograms of the projected mass
        # We bin the projected values to make Sinkhorn computation O(n_bins^2) 
        # instead of O(N_voxels^2)
        min_p, max_p = proj_coords.min(), proj_coords.max()
        bins = np.linspace(min_p, max_p, n_bins + 1)
        bin_centers = (bins[:-1] + bins[1:]) / 2
        
        # Calculate mass in each bin
        hist1, _ = np.histogram(proj_coords, bins=bins, weights=w1)
        hist2, _ = np.histogram(proj_coords, bins=bins, weights=w2)
        
        # Normalize histograms
        hist1 /= (hist1.sum() + 1e-12)
        hist2 /= (hist2.sum() + 1e-12)

        # 5. Compute 1D Cost Matrix (Squared Euclidean Distance)
        # Using bin centers for ground metric
        M = ot.dist(bin_centers.reshape(-1, 1), bin_centers.reshape(-1, 1), metric='sqeuclidean')
        M /= M.max() # Normalization for numerical stability

        # 6. Compute Sinkhorn Divergence
        # sinkhorn2 returns the distance (loss)
        dist = ot.sinkhorn2(hist1, hist2, M, reg)
        total_dist += dist

    return total_dist / n_projections

In [4]:

# Test 1: Standard Sliced Wasserstein (your current method)
start = time.time()
dist_sw = ot.sliced_wasserstein_distance(positions, positions, p, q, 
                                         n_projections=50, seed=42)
time_sw = time.time() - start
print(f"Sliced Wasserstein:        {dist_sw:.6f} in {time_sw:.2f}s")

# Test 2: Gemini's Sliced Sinkhorn with binning
start = time.time()
dist_ss = sliced_sinkhorn_3d(vol1, vol2, n_projections=50, reg=0.01, n_bins=512)
time_ss = time.time() - start
print(f"Sliced Sinkhorn (binned):  {dist_ss:.6f} in {time_ss:.2f}s")

# Test 3: Try fewer bins (faster?)
start = time.time()
dist_ss_256 = sliced_sinkhorn_3d(vol1, vol2, n_projections=50, reg=0.01, n_bins=256)
time_ss_256 = time.time() - start
print(f"Sliced Sinkhorn (n=256):   {dist_ss_256:.6f} in {time_ss_256:.2f}s")

print(f"\n{'='*60}")
print(f"Speedup (512 bins): {time_sw/time_ss:.2f}x")
print(f"Speedup (256 bins): {time_sw/time_ss_256:.2f}x")

Sliced Wasserstein:        0.343051 in 60.65s


/Users/MAlghalayini/Desktop/Postdoc Work/CAMERA gpLVM/Github Repository/Flexible_kernels/.venv/lib/python3.11/site-packages/ot/bregman/_sinkhorn.py:623: RuntimeWarning: divide by zero encountered in divide
  Kp = (1 / a).reshape(-1, 1) * K
/Users/MAlghalayini/Desktop/Postdoc Work/CAMERA gpLVM/Github Repository/Flexible_kernels/.venv/lib/python3.11/site-packages/ot/bregman/_sinkhorn.py:642: UserWarning: Warning: numerical errors at iteration 0
  warnings.warn("Warning: numerical errors at iteration %d" % ii)


Sliced Sinkhorn (binned):  0.000785 in 12.29s
Sliced Sinkhorn (n=256):   0.000784 in 11.76s

Speedup (512 bins): 4.93x
Speedup (256 bins): 5.16x


In [5]:
# Test on 3 volume pairs
pairs = [(vol1, vol2), (vol1, vol1), (vol2, vol2)]

print("Testing relative ordering...")
print(f"{'Pair':<15} {'Sliced Wass':<15} {'Sliced Sinkhorn':<15}")

for i, (v1, v2) in enumerate(pairs):
    # Sliced Wasserstein
    p = v1.flatten() / v1.sum()
    q = v2.flatten() / v2.sum()
    positions = np.array([[i, j, k] for i in range(128) for j in range(128) for k in range(128)])
    d_sw = ot.sliced_wasserstein_distance(positions, positions, p, q, n_projections=50, seed=42)
    
    # Sliced Sinkhorn
    d_ss = sliced_sinkhorn_3d(v1, v2, n_projections=50, reg=0.1, n_bins=256)  # Higher reg to avoid warnings
    
    print(f"Pair {i:<11} {d_sw:.6f}        {d_ss:.6f}")

Testing relative ordering...
Pair            Sliced Wass     Sliced Sinkhorn
Pair 0           0.343051        0.017994
Pair 1           0.000000        0.017994
Pair 2           0.000000        0.017994


In [6]:
# Test fewer projections on standard sliced Wasserstein
import time

results = []

for n_proj in [5, 10, 15, 20, 30, 50]:
    start = time.time()
    
    p = vol1.flatten() / vol1.sum()
    q = vol2.flatten() / vol2.sum()
    
    d = ot.sliced_wasserstein_distance(positions, positions, p, q, 
                                       n_projections=n_proj, seed=42)
    elapsed = time.time() - start
    
    results.append((n_proj, d, elapsed))
    print(f"n_proj={n_proj:2d}: dist={d:.6f}, time={elapsed:5.2f}s, speedup={60.65/elapsed:.2f}x")

# Check accuracy vs n=50
baseline_dist = results[-1][1]
print(f"\n{'='*60}")
print(f"Accuracy check (vs n=50={baseline_dist:.6f}):")
for n_proj, d, t in results[:-1]:
    error = abs(d - baseline_dist) / baseline_dist * 100
    print(f"n_proj={n_proj:2d}: error={error:.2f}%, speedup={60.65/t:.2f}x")

n_proj= 5: dist=0.241868, time= 4.07s, speedup=14.91x
n_proj=10: dist=0.327871, time= 8.29s, speedup=7.32x
n_proj=15: dist=0.342026, time=12.77s, speedup=4.75x
n_proj=20: dist=0.299116, time=19.25s, speedup=3.15x
n_proj=30: dist=0.319384, time=36.91s, speedup=1.64x
n_proj=50: dist=0.343051, time=55.98s, speedup=1.08x

Accuracy check (vs n=50=0.343051):
n_proj= 5: error=29.49%, speedup=14.91x
n_proj=10: error=4.42%, speedup=7.32x
n_proj=15: error=0.30%, speedup=4.75x
n_proj=20: error=12.81%, speedup=3.15x
n_proj=30: error=6.90%, speedup=1.64x
